# Face Swap Service — 1 người, dùng lại nhiều lần trong 1 session Colab

**Khác bản trước ở chỗ nào:** `USE_GFPGAN` không còn là công tắc lúc cài đặt nữa, mà là
**tham số mỗi lần gọi**. Notebook chia làm hai phần tách bạch:

| | Chạy khi nào | Làm gì | Tốn |
|---|---|---|---|
| **PHẦN A — SETUP** | **Một lần** cho mỗi session Colab | cài lib, vá source, tải model, **nạp model vào VRAM** | vài phút |
| **PHẦN B — HÀM** | Một lần (chỉ định nghĩa hàm) | định nghĩa `process_video(...)` | tức thì |
| **PHẦN C — DÙNG** | **Mỗi lần** có ảnh/video mới | gọi `process_video(...)` | chỉ thời gian xử lý |
| **PHẦN D — API** | Tùy chọn | mở tunnel cloudflared, nhận link ảnh/video qua HTTP | — |

## Vì sao đặt như vậy mới đúng

Bản trước đặt `USE_GFPGAN` ở đầu và cho nó điều khiển luôn cả bước cài đặt lẫn tải model.
Với cách dùng "chạy một lần rồi thôi" thì hợp lý — tắt đi thì đỡ tải 340 MB.

Nhưng với cách dùng **service** thì nó hỏng: session nào lỡ đặt `False` là không có `basicsr`,
không có `GFPGANv1.4.pth`, nên **request sau muốn làm nét sẽ chết** — muốn đổi ý phải cài lại
từ đầu, mất vài phút. Công tắc lúc build không thể trả lời một câu hỏi thuộc về lúc chạy.

Nên ở đây đảo lại:

- **Cài và tải VÔ ĐIỀU KIỆN.** Tốn thêm ~340 MB và ~1 phút, đúng một lần cho cả session.
- **Nạp GFPGAN vào VRAM luôn ở PHẦN A** (`RESTORER`), giữ đó. Nạp model tốn vài giây; làm sẵn
  thì lúc phục vụ, bật/tắt làm nét chỉ là một câu `if` — **miễn phí, không nạp lại gì cả**.
- **`process_video(..., use_gfpgan=True/False)`** quyết định từng lần gọi.

Đổi lại được: mỗi request là một lời gọi hàm với tham số riêng, đúng thứ một endpoint HTTP cần.

## Hai công tắc độc lập

Ngoài `use_gfpgan`, hàm còn nhận `do_swap` — vì bạn có nói muốn "có làm nét cũng được mà không
swap luôn cũng được". Hai cờ rời nhau phủ đủ 4 tổ hợp:

| `do_swap` | `use_gfpgan` | Kết quả |
|---|---|---|
| `True` | `True` | swap mặt + làm nét — mặc định |
| `True` | `False` | chỉ swap, nhanh nhất |
| `False` | `True` | **không swap, chỉ làm nét mặt gốc** (dùng để phục hồi video mờ) |
| `False` | `False` | chỉ re-encode, hàm sẽ cảnh báo vì gần như vô nghĩa |

⚠️ Lưu ý: công nghệ face-swap có thể bị dùng sai mục đích (deepfake giả mạo người khác mà
không có sự đồng ý). Chỉ dùng với ảnh/video của chính bạn hoặc người đã đồng ý, và cân nhắc
gắn watermark/disclosure khi xuất bản sản phẩm thật.

---
# PHẦN A — SETUP

**Chạy hết phần này đúng một lần** sau khi mở session Colab (Runtime > Change runtime type >
GPU T4). Xong rồi thì không đụng lại nữa, trừ khi Colab ngắt kết nối.

## A1. Cài thư viện

In [ ]:
# Log phiên bản Python Colab đang cấp (để biết đang chạy bản nào)
import sys, platform
print('Python version:', sys.version)
print('Platform:', platform.platform())

In [ ]:
# Ghim setuptools < 82: bản setuptools mới (>=82) đã bỏ hẳn module 'distutils',
# trong khi basicsr (dependency của GFPGAN) và torch trên Colab vẫn cần distutils/setuptools cũ.
!pip install -q "setuptools==79.0.1" wheel
!pip install -q cython numpy

# --no-build-isolation: để insightface dùng đúng cython/numpy vừa cài ở trên,
# thay vì pip tự tạo môi trường tạm cô lập (không thấy cython) rồi build lỗi 'egg_info'.
!pip install -q --no-build-isolation insightface==0.7.3

# KHÔNG cài opencv-python-headless: Colab đã có sẵn opencv-python, mà hai package này
# dùng chung namespace `cv2` -> cài đè lên nhau hay để lại .so lẫn lộn gây lỗi khó hiểu.

# gfpgan/facexlib cài VÔ ĐIỀU KIỆN: việc bật/tắt làm nét là quyết định lúc CHẠY,
# không phải lúc cài. Cài sẵn thì mỗi request tự chọn được, không phải cài lại.
!pip install -q onnxruntime-gpu gfpgan facexlib

!apt-get -qq install -y ffmpeg > /dev/null
print('Xong.')

## A2. Vá `basicsr` (bug PEP 667 ở Python 3.13)

`basicsr` có bug trong `setup.py`: dùng `exec(...)` rồi đọc `locals()['__version__']` để lấy
version. Ở Python 3.13, `exec()` trong 1 hàm không còn ghi ngược lại `locals()` đáng tin cậy
(thay đổi theo PEP 667) → lỗi `KeyError: '__version__'`. Cell dưới tải source về, patch đúng
chỗ này (`locals()` → `globals()`), rồi cài từ bản đã sửa.

In [ ]:
import subprocess, sys, os, tarfile, urllib.request, json


def install_basicsr_patched():
    """Tải basicsr từ PyPI, vá bug PEP 667 trong setup.py, rồi cài từ source đã sửa."""
    os.makedirs('/tmp/basicsr_src', exist_ok=True)

    # Tải trực tiếp từ PyPI bằng urllib (KHÔNG dùng `pip download`, vì pip cũng phải chạy
    # setup.py egg_info để lấy metadata -> dính đúng bug KeyError trước khi kịp patch).
    with urllib.request.urlopen('https://pypi.org/pypi/basicsr/1.4.2/json') as resp:
        pkg_info = json.load(resp)

    sdist_url = None
    for url_info in pkg_info['urls']:
        if url_info['packagetype'] == 'sdist':
            sdist_url = url_info['url']
            break
    assert sdist_url is not None, 'Không tìm thấy sdist của basicsr trên PyPI.'

    tar_name = sdist_url.split('/')[-1]
    tar_path = f'/tmp/basicsr_src/{tar_name}'
    urllib.request.urlretrieve(sdist_url, tar_path)
    print(f'Đã tải: {tar_name}')

    extract_dir = '/tmp/basicsr_build'
    os.makedirs(extract_dir, exist_ok=True)
    with tarfile.open(tar_path) as tar:
        # Lấy tên thư mục gốc từ chính nội dung tar thay vì suy ra bằng tar_name.replace(...),
        # vì sdist trên PyPI không bắt buộc phải là .tar.gz.
        root_names = {m.name.split('/')[0] for m in tar.getmembers() if m.name.strip('./')}
        assert len(root_names) == 1, f'sdist có cấu trúc thư mục lạ: {root_names}'
        root_name = root_names.pop()
        # filter='data': Python 3.12+ deprecate extractall không có filter, 3.14 đổi default.
        tar.extractall(extract_dir, filter='data')

    pkg_dir = os.path.join(extract_dir, root_name)
    setup_py_path = os.path.join(pkg_dir, 'setup.py')
    assert os.path.exists(setup_py_path), f'Không thấy setup.py trong {pkg_dir}'

    with open(setup_py_path, 'r') as f:
        content = f.read()

    # Fix bug Python 3.13: exec() trong hàm không ghi ngược locals() đáng tin cậy
    content = content.replace(
        "exec(compile(f.read(), version_file, 'exec'))",
        "exec(compile(f.read(), version_file, 'exec'), globals())"
    )
    content = content.replace(
        "return locals()['__version__']",
        "return globals()['__version__']"
    )

    with open(setup_py_path, 'w') as f:
        f.write(content)

    print('Đã patch setup.py xong, tiến hành cài basicsr từ source đã sửa...')
    # sys.executable -m pip: đảm bảo cài vào đúng interpreter đang chạy notebook,
    # thay vì lệnh `pip` bất kỳ đứng đầu PATH.
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--no-build-isolation', pkg_dir],
        capture_output=True, text=True
    )
    print(r.stdout[-3000:])
    print(r.stderr[-3000:])
    if r.returncode != 0:
        raise RuntimeError('Cài basicsr thất bại, xem log lỗi ở trên để biết nguyên nhân cụ thể.')
    print('Cài basicsr thành công.')


install_basicsr_patched()

## A3. Vá tham chiếu `torchvision.transforms.functional_tensor`

Bản `torchvision` mới đã xoá hẳn module `torchvision.transforms.functional_tensor`, trong khi
`basicsr` vẫn import `rgb_to_grayscale` theo đường cũ đó → `ModuleNotFoundError`. Hàm này nay
nằm ở `torchvision.transforms.functional`.

Cell dưới định vị package bằng `importlib.util.find_spec` — hàm này chỉ **tìm** package chứ
không chạy `__init__.py`, nên không dính đúng cái lỗi import mà ta đang muốn vá — rồi sửa mọi
file `.py` còn tham chiếu module cũ. Nó cũng tự xoá module hỏng khỏi `sys.modules` cả trước
lẫn sau khi vá, nhờ vậy **không cần Runtime > Restart session**.

In [ ]:
import importlib, importlib.util, sys, pathlib

OLD_MOD = 'torchvision.transforms.functional_tensor'
NEW_MOD = 'torchvision.transforms.functional'
PKGS = ('basicsr', 'facexlib', 'gfpgan')


def purge_modules():
    """Xoá các module (có thể đã import hỏng) khỏi cache.

    Phải chạy TRƯỚC find_spec: một lần import hỏng trước đó có thể để lại
    sys.modules['basicsr'] với __spec__ = None, khiến find_spec ném ValueError
    chứ không trả về None -> cell báo nhầm "chưa cài" dù basicsr đang có trên đĩa.
    """
    for mod in [m for m in list(sys.modules) if m.split('.')[0] in PKGS]:
        del sys.modules[mod]
    importlib.invalidate_caches()


def package_dir(name):
    """Trả về thư mục package mà KHÔNG import nó."""
    try:
        spec = importlib.util.find_spec(name)
    except Exception as e:
        print(f'  (không tra được {name}: {type(e).__name__}: {e})')
        return None
    if spec is None or not spec.submodule_search_locations:
        return None
    return pathlib.Path(list(spec.submodule_search_locations)[0])


def patch_torchvision_refs():
    purge_modules()

    targets = {}
    for name in PKGS:
        d = package_dir(name)
        if d is None:
            print(f'{name:9s}: CHƯA CÀI')
        else:
            print(f'{name:9s}: {d}')
            targets[name] = d

    assert 'basicsr' in targets, (
        'Không tìm thấy basicsr. Chạy lại cell A2 rồi chạy lại cell này.'
    )

    patched = []
    for name, d in targets.items():
        for p in d.rglob('*.py'):
            try:
                text = p.read_text(encoding='utf-8')
            except (UnicodeDecodeError, OSError):
                continue
            if OLD_MOD not in text:
                continue
            p.write_text(text.replace(OLD_MOD, NEW_MOD), encoding='utf-8')
            patched.append(p)

    print()
    if patched:
        for p in patched:
            print(f'đã vá: {p}')
    else:
        print('Không file nào cần vá (đã vá trước đó, hoặc torchvision bản này vẫn còn functional_tensor).')

    purge_modules()   # để lần import sau đọc lại file mới trên đĩa

    try:
        import basicsr.data.degradations
        print()
        print('OK: import basicsr.data.degradations thành công.')
    except Exception as e:
        print()
        print(f'VẪN LỖI: {type(e).__name__}: {e}')
        print('Nếu lỗi vẫn liên quan tới torchvision, thử Runtime > Restart session rồi chạy lại.')
        raise


patch_torchvision_refs()

## A4. Tải model

Tải **cả ba**, không điều kiện: `buffalo_l` (insightface tự tải lúc `prepare`), `inswapper_128`,
và `GFPGANv1.4`. Tải sẵn GFPGAN dù request hiện tại không dùng — vì request sau có thể dùng.

In [ ]:
import os, subprocess

MODEL_DIR = '/content/models'
os.makedirs(MODEL_DIR, exist_ok=True)

INSWAPPER_PATH = f'{MODEL_DIR}/inswapper_128.onnx'
GFPGAN_PATH = f'{MODEL_DIR}/GFPGANv1.4.pth'

INSWAPPER_URL = 'https://huggingface.co/ezioruan/inswapper_128.onnx/resolve/main/inswapper_128.onnx'
GFPGAN_URL = 'https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/GFPGANv1.4.pth'


def download(url, path, min_mb, hint):
    """Tải file rồi KIỂM TRA DUNG LƯỢNG.

    wget vẫn coi là "thành công" khi server trả 404 / trang HTML -> phải tự kiểm tra.
    Nếu không, mãi tới cell nạp model mới nổ với lỗi onnx/torch rất khó đoán nguyên nhân.
    Bỏ qua nếu file đã có sẵn và đủ lớn -> chạy lại cell này không tải lại từ đầu.
    """
    if os.path.exists(path) and os.path.getsize(path) / 1e6 >= min_mb:
        print(f'BỎ QUA  {os.path.basename(path)}: đã có ({os.path.getsize(path) / 1e6:.1f} MB)')
        return
    print(f'Đang tải {os.path.basename(path)} ...')
    subprocess.run(['wget', '-q', '-O', path, url])
    size_mb = os.path.getsize(path) / 1e6 if os.path.exists(path) else 0
    assert size_mb >= min_mb, (
        f'Tải {os.path.basename(path)} thất bại (chỉ {size_mb:.1f} MB, cần >= {min_mb} MB). {hint}'
    )
    print(f'OK      {os.path.basename(path)}: {size_mb:.1f} MB')


download(INSWAPPER_URL, INSWAPPER_PATH, 200,
         'Link mirror có thể đã chết -> tự tải rồi upload vào /content/models/.')
download(GFPGAN_URL, GFPGAN_PATH, 300,
         'Kiểm tra lại link GitHub release của GFPGAN.')

!ls -lh /content/models/

## A5. Nạp model vào bộ nhớ — **chỗ đắt nhất, và là lý do của cả thiết kế này**

Ba model (`buffalo_l` detect, `inswapper` swap, `GFPGAN` làm nét) được nạp vào VRAM **một lần**
rồi giữ nguyên trong biến toàn cục suốt session. Mỗi lần xử lý video mới chỉ là gọi hàm dùng
lại chúng — không nạp lại gì.

**GFPGAN nạp kể cả khi request đầu tiên không cần**, tốn ~350 MB VRAM nằm chờ (T4 có 15 GB nên
không đáng kể). Đổi lại, `use_gfpgan=True/False` trở thành một câu `if` miễn phí thay vì phải
nạp model mất vài giây giữa lúc đang phục vụ.

Nếu GFPGAN nạp hỏng, `RESTORER = None` và mọi thứ vẫn chạy — chỉ là các request xin làm nét sẽ
được trả về kèm cảnh báo, thay vì làm sập server.

In [ ]:
import subprocess, sys, importlib


def try_import_onnxruntime():
    # Bắt Exception chứ không chỉ ImportError: onnxruntime-gpu thiếu libcudnn/libcublas
    # thường ném OSError/RuntimeError -> nếu chỉ bắt ImportError thì cell crash thay vì fallback.
    importlib.invalidate_caches()
    try:
        import onnxruntime
        print('onnxruntime OK, version:', onnxruntime.__version__)
        print('Available providers:', onnxruntime.get_available_providers())
        return True
    except Exception as e:
        print(f'Chưa import được onnxruntime: {type(e).__name__}: {e}')
        return False


def pip(*args):
    r = subprocess.run([sys.executable, '-m', 'pip', *args], capture_output=True, text=True)
    print(r.stdout[-3000:])
    print(r.stderr[-3000:])
    return r.returncode


if not try_import_onnxruntime():
    # onnxruntime và onnxruntime-gpu cùng chiếm package `onnxruntime`. Cài cái này đè cái kia
    # là trạng thái hỏng đã biết -> luôn gỡ sạch cả hai trước khi cài lại.
    print('Gỡ sạch onnxruntime cũ rồi cài lại onnxruntime-gpu...')
    pip('uninstall', '-y', 'onnxruntime', 'onnxruntime-gpu')
    pip('install', 'onnxruntime-gpu==1.20.0')

    if not try_import_onnxruntime():
        print('onnxruntime-gpu không cài được (khả năng chưa có wheel cho bản Python này).')
        print('Fallback sang onnxruntime bản CPU...')
        pip('uninstall', '-y', 'onnxruntime', 'onnxruntime-gpu')
        pip('install', 'onnxruntime')
        assert try_import_onnxruntime(), 'Vẫn không cài được onnxruntime, xem log lỗi ở trên.'

In [ ]:
import cv2
import insightface
from insightface.app import FaceAnalysis
import onnxruntime

# ---------- detect + swap ----------
available_providers = onnxruntime.get_available_providers()
USE_CUDA = 'CUDAExecutionProvider' in available_providers
providers = ['CUDAExecutionProvider', 'CPUExecutionProvider'] if USE_CUDA else ['CPUExecutionProvider']
# ctx_id phải khớp với providers: 0 = GPU 0, -1 = CPU. Để nguyên 0 khi chỉ có CPU provider là mâu thuẫn.
ctx_id = 0 if USE_CUDA else -1
print('Providers:', providers, '| ctx_id =', ctx_id)

APP = FaceAnalysis(name='buffalo_l', providers=providers)
APP.prepare(ctx_id=ctx_id, det_size=(640, 640))

SWAPPER = insightface.model_zoo.get_model(INSWAPPER_PATH, download=False, providers=providers)
print('APP + SWAPPER đã nạp.')

# ---------- làm nét (nạp sẵn, dùng hay không là chuyện của từng request) ----------
RESTORER = None
try:
    from gfpgan import GFPGANer
    RESTORER = GFPGANer(
        model_path=GFPGAN_PATH,
        upscale=1,
        arch='clean',
        channel_multiplier=2,
        bg_upsampler=None
    )
    print('RESTORER (GFPGAN) đã nạp — mỗi request tự chọn dùng hay không.')
except Exception as e:
    # Không raise: thiếu GFPGAN chỉ làm mất tính năng làm nét, không được phép làm sập
    # cả service. Request nào xin làm nét sẽ nhận cảnh báo trong kết quả trả về.
    print(f'KHÔNG nạp được GFPGAN: {type(e).__name__}: {e}')
    print('-> Service vẫn chạy, nhưng use_gfpgan sẽ bị bỏ qua.')

print()
print('=' * 60)
print('SẴN SÀNG')
print(f'  GPU        : {"CÓ" if USE_CUDA else "KHÔNG (chạy CPU, sẽ chậm)"}')
print(f'  Làm nét    : {"CÓ" if RESTORER is not None else "KHÔNG"}')
print('=' * 60)

---
# PHẦN B — Hàm xử lý

Chạy hai cell dưới **một lần** để định nghĩa hàm. Từ đó về sau, mỗi ảnh/video mới chỉ cần gọi:

```python
process_video(face_src, video_src, use_gfpgan=True, do_swap=True)
```

`face_src` và `video_src` nhận **cả URL lẫn đường dẫn file local** — nên phần API ở dưới không
cần code đường riêng, cứ đưa thẳng link vào.

In [ ]:
import os, time, uuid, urllib.request, urllib.parse
import numpy as np
import cv2

WORK_DIR = '/content/work'
OUT_DIR = '/content/out'
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

_req_counter = 0


def next_job_id():
    """ID duy nhất cho mỗi lần xử lý, để output không đè lên nhau.

    Timestamp + counter thôi là chưa đủ: chạy lại cell này sẽ reset counter về 0,
    và hai job trong cùng một giây sẽ trùng ID -> job sau ghi đè kết quả job trước.
    Với service thì đó là mất kết quả của người khác, nên thêm uuid cho chắc.
    """
    global _req_counter
    _req_counter += 1
    return f'{int(time.time())}_{_req_counter:03d}_{uuid.uuid4().hex[:6]}'


def fetch(src, job_id, kind):
    """Nhận URL hoặc đường dẫn local, trả về đường dẫn local.

    Giả lập trình duyệt để qua mặt chặn hotlink/403: nhiều host (kể cả HuggingFace,
    Google Drive, CDN ảnh) chặn thẳng 'Python-urllib/3.x' và trả về 403 hoặc một
    trang HTML — mà HTML ghi ra file .mp4 thì cv2 chỉ báo "không mở được video",
    không nói vì sao. Referer trỏ về chính domain đó là mẹo qua được phần lớn
    kiểm tra hotlink.
    """
    src = str(src).strip()
    if not src.lower().startswith(('http://', 'https://')):
        assert os.path.exists(src), f'Không thấy file {kind}: {src}'
        return src

    sp = urllib.parse.urlsplit(src)
    headers = {
        'User-Agent': ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                       '(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36'),
        'Accept': 'image/avif,image/webp,image/*,video/*,*/*;q=0.8',
        'Referer': f'{sp.scheme}://{sp.netloc}/',
    }

    ext = os.path.splitext(sp.path)[1][:8] or ''
    path = os.path.join(WORK_DIR, f'{job_id}_{kind}{ext}')
    req = urllib.request.Request(src, headers=headers)
    with urllib.request.urlopen(req, timeout=180) as r, open(path, 'wb') as f:
        while True:
            chunk = r.read(1 << 20)
            if not chunk:
                break
            f.write(chunk)

    size = os.path.getsize(path)
    assert size > 1024, f'Tải {kind} về chỉ được {size} bytes — link hỏng hoặc bị chặn: {src}'
    return path


def load_source_face(path):
    """Đọc ảnh nguồn và lấy khuôn mặt lớn nhất."""
    # cv2.imread trả None khi file hỏng / định dạng không hỗ trợ (heic, webp lạ...).
    # Phải chặn ngay, không thì APP.get(None) ném lỗi cv2 không nói gì về nguyên nhân thật.
    img = cv2.imread(path)
    assert img is not None, f'Không đọc được ảnh nguồn: {path}. Dùng .jpg/.png.'
    faces = APP.get(img)
    assert len(faces) > 0, 'Không tìm thấy khuôn mặt trong ảnh nguồn, thử ảnh khác rõ mặt hơn.'
    # Thứ tự APP.get() trả về KHÔNG xác định -> không được lấy faces[0].
    return max(faces, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]))


def pick_face(faces, min_det_score):
    """Chọn khuôn mặt để xử lý trong một frame. None nếu không có mặt nào đủ tốt.

    Video 1 người nên không cần tracking theo danh tính: mặt LỚN NHẤT là chủ thể.
    KHÔNG dùng faces[0] vì thứ tự APP.get() trả về không xác định — video có người đi
    ngang trong nền thì faces[0] nhảy qua nhảy lại giữa các frame.
    """
    cands = [f for f in faces if f.det_score >= min_det_score]
    if not cands:
        return None
    return max(cands, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]))


def enhance_face_region(img, face, pad):
    """Chỉ làm nét vùng quanh khuôn mặt, KHÔNG chạy trên cả frame.

    Gọi RESTORER.enhance() trên nguyên frame khiến GFPGAN chạy lại một bộ face detector
    thứ hai trên toàn khung (trùng lặp với APP.get() vừa chạy), làm nét luôn cả những mặt
    trong nền không liên quan, và resize LANCZOS toàn frame mỗi lượt.
    """
    h, w = img.shape[:2]
    x1, y1, x2, y2 = face.bbox
    bw, bh = x2 - x1, y2 - y1
    x1 = max(0, int(x1 - pad * bw)); x2 = min(w, int(x2 + pad * bw))
    y1 = max(0, int(y1 - pad * bh)); y2 = min(h, int(y2 + pad * bh))
    cw, ch = x2 - x1, y2 - y1
    if cw < 64 or ch < 64:
        return img

    crop = img[y1:y2, x1:x2]
    _, _, restored = RESTORER.enhance(crop, has_aligned=False,
                                      only_center_face=True, paste_back=True)
    if restored is None:
        return img
    if restored.shape[:2] != (ch, cw):
        restored = cv2.resize(restored, (cw, ch), interpolation=cv2.INTER_LANCZOS4)

    # Blend viền mềm để vùng crop không để lại đường viền hình chữ nhật thấy rõ.
    mask = np.zeros((ch, cw), np.float32)
    cv2.rectangle(mask, (int(0.12 * cw), int(0.12 * ch)),
                  (int(0.88 * cw), int(0.88 * ch)), 1.0, -1)
    mask = cv2.GaussianBlur(mask, (0, 0), sigmaX=max(2.0, 0.05 * max(cw, ch)))[..., None]
    img[y1:y2, x1:x2] = (restored * mask + crop * (1.0 - mask)).astype(np.uint8)
    return img


print('Helpers sẵn sàng.')

In [ ]:
import subprocess
from tqdm.auto import tqdm


def process_video(face_src, video_src, out_path=None,
                  use_gfpgan=True, do_swap=True,
                  min_det_score=0.5, gfpgan_pad=0.4, progress=True):
    """Xử lý một video. Dùng lại model đã nạp ở PHẦN A, không nạp lại gì.

    face_src   : URL hoặc path ảnh khuôn mặt nguồn (bỏ qua nếu do_swap=False)
    video_src  : URL hoặc path video
    use_gfpgan : có làm nét mặt sau khi xử lý không
    do_swap    : có thay mặt không. False + use_gfpgan=True = chỉ làm nét mặt gốc.

    Trả về dict thống kê, trong đó 'output' là đường dẫn file kết quả.
    """
    job_id = next_job_id()
    t_start = time.perf_counter()
    warnings = []

    if not do_swap and not use_gfpgan:
        warnings.append('do_swap=False và use_gfpgan=False: video ra chỉ là bản re-encode.')

    # RESTORER là None khi GFPGAN nạp hỏng ở A5 -> hạ cấp có báo, không ném lỗi.
    restorer_on = use_gfpgan and RESTORER is not None
    if use_gfpgan and RESTORER is None:
        warnings.append('use_gfpgan=True nhưng GFPGAN không nạp được -> bỏ qua bước làm nét.')

    video_path = fetch(video_src, job_id, 'video')
    source_face = None
    if do_swap:
        source_face = load_source_face(fetch(face_src, job_id, 'face'))

    if out_path is None:
        out_path = os.path.join(OUT_DIR, f'{job_id}.mp4')

    cap = cv2.VideoCapture(video_path)
    assert cap.isOpened(), f'Không mở được video: {video_path}'

    fps = cap.get(cv2.CAP_PROP_FPS)
    if not fps or fps != fps or fps <= 0:      # 0.0 hoặc NaN với một số container
        warnings.append('Không đọc được fps từ video, dùng mặc định 25.')
        fps = 25.0

    # CAP_PROP_FRAME_COUNT chỉ là ước lượng (sai với VFR / mp4 thiếu index) -> chỉ dùng cho
    # progress bar. Vòng lặp đọc tới khi hết frame thật, thay vì range(total) (đếm thiếu =
    # cụt đuôi video).
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or None

    # Lấy kích thước từ frame THẬT, không từ CAP_PROP_FRAME_WIDTH/HEIGHT: video quay dọc có
    # metadata rotation làm hai giá trị đó bị hoán đổi so với frame OpenCV trả về -> ffmpeg
    # nhận rawvideo sai size.
    ret, frame = cap.read()
    assert ret, 'Không đọc được frame nào từ video.'
    height, width = frame.shape[:2]

    # Ghi frame thô thẳng vào ffmpeg và ghép audio ngay trong cùng một pass.
    ffmpeg_cmd = [
        'ffmpeg', '-y', '-loglevel', 'error',
        '-f', 'rawvideo', '-pix_fmt', 'bgr24', '-s', f'{width}x{height}',
        '-r', f'{fps}', '-i', 'pipe:0',
        '-i', video_path,
        '-map', '0:v:0', '-map', '1:a:0?',     # '?' = không có audio thì bỏ qua, không lỗi
        '-c:v', 'libx264', '-crf', '18', '-preset', 'fast',
        '-pix_fmt', 'yuv420p',                 # để trình duyệt phát được
        '-c:a', 'aac', '-shortest',
        out_path,
    ]
    proc = subprocess.Popen(ffmpeg_cmd, stdin=subprocess.PIPE)

    # Chỉ cần detect khi có việc để làm với khuôn mặt.
    need_faces = do_swap or restorer_on

    pbar = tqdm(total=total_frames, unit='frame', disable=not progress)
    n_written = n_faces = 0
    t_detect = t_swap = t_enhance = 0.0
    rc = None
    try:
        while frame is not None:
            result = frame

            if need_faces:
                t0 = time.perf_counter()
                face = pick_face(APP.get(frame), min_det_score)
                t1 = time.perf_counter()
                t_detect += t1 - t0

                if face is not None:
                    n_faces += 1
                    if do_swap:
                        result = SWAPPER.get(frame, face, source_face, paste_back=True)
                    t2 = time.perf_counter()
                    t_swap += t2 - t1
                    if restorer_on:
                        result = enhance_face_region(result, face, gfpgan_pad)
                        t_enhance += time.perf_counter() - t2

            proc.stdin.write(np.ascontiguousarray(result).tobytes())
            n_written += 1
            pbar.update(1)

            ret, frame = cap.read()
            if not ret:
                frame = None
    finally:
        # Không đóng trong finally thì khi bấm Stop giữa chừng, ffmpeg treo và mp4 hỏng.
        pbar.close()
        cap.release()
        try:
            proc.stdin.close()
        except BrokenPipeError:
            pass
        rc = proc.wait()

    assert rc == 0, f'ffmpeg thất bại (exit code {rc}).'

    wall = time.perf_counter() - t_start
    n = max(n_written, 1)
    nf = max(n_faces, 1)
    return {
        'job_id': job_id,
        'output': out_path,
        'kich_thuoc_mb': round(os.path.getsize(out_path) / 1e6, 2),
        'do_swap': do_swap,
        'lam_net': restorer_on,
        'so_frame': n_written,
        'so_frame_co_mat': n_faces,
        'ty_le_co_mat': round(n_faces / n, 3),
        'ms_detect': round(t_detect / n * 1000, 1),
        'ms_swap': round(t_swap / nf * 1000, 1),
        'ms_lam_net': round(t_enhance / nf * 1000, 1),
        'tong_giay': round(wall, 1),
        'fps_xu_ly': round(n / max(wall, 1e-9), 2),
        'canh_bao': warnings,
    }


def show_result(r):
    """In kết quả process_video cho dễ đọc."""
    for k, v in r.items():
        if k == 'canh_bao':
            continue
        print(f'  {k:16s}: {v}')
    for w in r['canh_bao']:
        print(f'  CẢNH BÁO: {w}')


print('process_video() sẵn sàng.')

---
# PHẦN C — Dùng tay

**Đây là phần chạy lại nhiều lần.** Mỗi ảnh/video mới chỉ cần chạy lại mấy cell dưới —
PHẦN A và B đã xong rồi, model vẫn nằm sẵn trong VRAM.

In [ ]:
from google.colab import files


def upload_one(what):
    print(f'>> Upload {what}:')
    up = files.upload()
    names = list(up.keys())
    assert names, f'Chưa upload {what}. Chạy lại cell này.'
    if len(names) > 1:
        print(f'  (đã upload {len(names)} file, dùng file đầu tiên: {names[0]})')
    return names[0]


face_path = upload_one('ảnh khuôn mặt (rõ mặt, chính diện càng tốt)')
print()
video_path = upload_one('video mẫu')
print()
print('Ảnh :', face_path)
print('Video:', video_path)

In [ ]:
# ===== Chỉnh hai cờ này rồi chạy lại cell, không cần đụng gì ở PHẦN A =====
result = process_video(
    face_path,
    video_path,
    use_gfpgan=True,     # False = bỏ làm nét, nhanh hơn nhiều
    do_swap=True,        # False = không thay mặt (kết hợp use_gfpgan=True để chỉ làm nét)
)
print()
show_result(result)

In [ ]:
import os
from IPython.display import Video, display

path = result['output']
size_mb = os.path.getsize(path) / 1e6
if size_mb > 50:
    # embed=True nhét cả file dưới dạng base64 vào output notebook -> file .ipynb phình to
    # và trình duyệt dễ treo với video dài.
    print(f'Video {size_mb:.1f} MB — quá lớn để nhúng inline, chạy cell dưới để tải về.')
else:
    display(Video(path, embed=True, width=480))

In [ ]:
# Tải file về máy
from google.colab import files
files.download(result['output'])

### So sánh bật/tắt làm nét trên đúng video của bạn

Cell dưới chạy cùng một video hai lần, chỉ khác `use_gfpgan`, rồi in bảng so sánh. Chạy với
video ngắn (5–10s) là đủ để biết làm nét đắt cỡ nào trên máy đang được cấp.

In [ ]:
r_off = process_video(face_path, video_path, use_gfpgan=False, progress=False)
r_on = process_video(face_path, video_path, use_gfpgan=True, progress=False)

print(f'{"":16s} {"TẮT làm nét":>14s} {"BẬT làm nét":>14s}')
for k in ('ms_detect', 'ms_swap', 'ms_lam_net', 'fps_xu_ly', 'tong_giay', 'kich_thuoc_mb'):
    print(f'{k:16s} {r_off[k]:>14} {r_on[k]:>14}')

if r_on['tong_giay'] > 0 and r_off['tong_giay'] > 0:
    print()
    print(f'-> Bật làm nét chậm hơn {r_on["tong_giay"] / r_off["tong_giay"]:.1f} lần.')

---
# PHẦN D — Server + tunnel (cloudflared)

Cấu trúc ở trên đã sẵn sàng cho phần này: endpoint chỉ việc gọi `process_video(...)`,
**không phải nạp lại model** vì chúng đã nằm sẵn trong biến toàn cục từ PHẦN A.

## Ba quyết định thiết kế, và lý do

**1. `cloudflared` thay vì `ngrok` — không cần tài khoản.** Chỉ tải một binary rồi chạy
`cloudflared tunnel --url http://localhost:8000`, nó tự in ra URL `*.trycloudflare.com`.
Không authtoken, không giới hạn số tunnel, không có trang cảnh báo chèn giữa như ngrok free.

**2. `/swap` chạy BẤT ĐỒNG BỘ — đây là điểm quan trọng nhất.** Cloudflare cắt kết nối ở
khoảng **100 giây** (lỗi 524). Swap một video 30 giây có thể mất vài phút, nên nếu render
đồng bộ thì **request nào cũng chết giữa chừng** — mà kết quả thì vẫn render xong ở server,
chỉ là client không bao giờ nhận được. Vì vậy:

```
POST /swap          -> trả NGAY {job_id, status: "processing"}
GET  /jobs/{id}     -> poll tới khi status = "done" (kèm thống kê)
GET  /jobs/{id}/result -> tải file mp4 về
```

Mỗi request đều trả trong mili-giây nên không bao giờ chạm trần timeout.

**3. `RUN_LOCK` — chỉ một job chạy trên GPU tại một thời điểm.** Đây chính là vấn đề tôi đã
nêu ở lần trước: `APP` / `SWAPPER` / `RESTORER` không an toàn khi gọi song song, mà FastAPI
lại chạy handler `def` trong threadpool nên hai request cùng lúc sẽ đụng nhau thật. Lock giải
quyết triệt để: job đến sau xếp hàng đợi, không bị từ chối.

Kèm theo, `JOB_TTL` tự dọn file kết quả sau 1 giờ — cũng là vấn đề đầy đĩa đã nêu.

**Cell cuối chạy mãi** để giữ tunnel sống (và giữ luôn phiên Colab khỏi bị ngắt vì idle).
Đợi dòng `PUBLIC_URL=...` hiện ra là dùng được. Bấm Stop để tắt.

In [ ]:
!pip install -q fastapi "uvicorn[standard]"

# Tải binary cloudflared (không cần tài khoản, không cần authtoken)
import os, stat, subprocess

CLOUDFLARED = '/usr/local/bin/cloudflared'
CF_URL = ('https://github.com/cloudflare/cloudflared/releases/latest/download/'
          'cloudflared-linux-amd64')

if not os.path.exists(CLOUDFLARED):
    print('Đang tải cloudflared...')
    subprocess.run(['wget', '-q', CF_URL, '-O', CLOUDFLARED], check=True)
    os.chmod(CLOUDFLARED, os.stat(CLOUDFLARED).st_mode | stat.S_IEXEC)

v = subprocess.run([CLOUDFLARED, '--version'], capture_output=True, text=True)
print('cloudflared:', (v.stdout or v.stderr).strip())

In [ ]:
import os, re, time, threading, subprocess, uuid, traceback
from typing import Optional
from fastapi import FastAPI, HTTPException
from fastapi.responses import FileResponse
from pydantic import BaseModel
import uvicorn

PORT = 8000
api = FastAPI(title='Face Swap Service')


class Job(BaseModel):
    video_url: str
    face_url: str = ''
    use_gfpgan: bool = True
    do_swap: bool = True
    job_id: Optional[str] = None   # client tự đặt id để poll bằng chính id đó


# ===== JOB STORE (bất đồng bộ) ==========================================
# /swap KHÔNG render đồng bộ. Lý do: Cloudflare cắt kết nối ở ~100s (lỗi 524),
# mà swap một video 30s có thể mất vài phút -> request nào cũng chết giữa chừng
# dù server vẫn render xong. Thay vào đó: tạo job_id, trả NGAY, render ở thread
# nền, client poll /jobs/{id} rồi tải ở /jobs/{id}/result.
JOBS = {}
JOBS_LOCK = threading.Lock()   # bảo vệ dict JOBS
RUN_LOCK = threading.Lock()    # serialize GPU: mỗi lúc chỉ 1 job được chạy
JOB_TTL = 3600                 # giữ kết quả 1 giờ rồi dọn file + entry


def _prune_jobs_locked():
    """Xoá job quá hạn kèm file kết quả. Gọi khi đang giữ JOBS_LOCK."""
    now = time.time()
    for k in [k for k, v in JOBS.items() if now - v.get('created', now) > JOB_TTL]:
        v = JOBS.pop(k, None)
        try:
            if v and v.get('output') and os.path.exists(v['output']):
                os.remove(v['output'])
        except OSError:
            pass


def _new_job(jid=None):
    jid = jid or uuid.uuid4().hex
    with JOBS_LOCK:
        _prune_jobs_locked()
        JOBS[jid] = {'status': 'processing', 'output': None, 'stats': None,
                     'error': None, 'created': time.time()}
    return jid


def _set_done(jid, stats):
    with JOBS_LOCK:
        if jid in JOBS:
            JOBS[jid].update(status='done', output=stats['output'], stats=stats)


def _set_error(jid, msg):
    with JOBS_LOCK:
        if jid in JOBS:
            JOBS[jid].update(status='error', error=str(msg)[:500])


def _run_async(jid, work):
    try:
        # RUN_LOCK: APP/SWAPPER/RESTORER dùng chung và KHÔNG an toàn khi gọi song song.
        # FastAPI chạy handler `def` trong threadpool nên hai request cùng lúc sẽ chạy
        # thật sự song song -> phải xếp hàng ở đây.
        with RUN_LOCK:
            stats = work()
        _set_done(jid, stats)
        print(f'[server] xong job {jid[:8]} -> {stats["output"]}')
    except Exception as e:
        traceback.print_exc()
        _set_error(jid, e)
        print(f'[server] LỖI job {jid[:8]}: {e}')


# ===== Endpoints ========================================================
@api.get('/health')
def health():
    """Cho client biết service làm nét được không TRƯỚC khi gửi việc."""
    with JOBS_LOCK:
        dang_cho = sum(1 for v in JOBS.values() if v['status'] == 'processing')
    return {
        'ok': True,
        'gpu': USE_CUDA,
        'lam_net_kha_dung': RESTORER is not None,
        'job_dang_chay': dang_cho,
    }


@api.post('/swap')
def swap(job: Job):
    if job.do_swap and not job.face_url:
        raise HTTPException(400, 'do_swap=true thì bắt buộc phải có face_url.')
    jid = _new_job(job.job_id)
    print(f'[server] nhận /swap ({jid[:8]}) -> chạy nền, trả job_id ngay.')

    def work():
        return process_video(job.face_url, job.video_url,
                             use_gfpgan=job.use_gfpgan, do_swap=job.do_swap,
                             progress=False)

    threading.Thread(target=_run_async, args=(jid, work), daemon=True).start()
    return {'job_id': jid, 'status': 'processing'}


@api.get('/jobs/{job_id}')
def job_status(job_id: str):
    with JOBS_LOCK:
        j = JOBS.get(job_id)
        if not j:
            raise HTTPException(404, 'job not found')
        return {
            'job_id': job_id,
            'status': j['status'],
            'error': j['error'],
            'thong_ke': j['stats'],
            'download_url': f'/jobs/{job_id}/result' if j['status'] == 'done' else None,
        }


@api.get('/jobs/{job_id}/result')
def job_result(job_id: str):
    with JOBS_LOCK:
        j = JOBS.get(job_id)
        if not j:
            raise HTTPException(404, 'job not found')
        status, path, err = j['status'], j['output'], j['error']
    if status == 'processing':
        raise HTTPException(409, 'job chưa xong')
    if status == 'error':
        raise HTTPException(500, err or 'job error')
    if not path or not os.path.exists(path):
        raise HTTPException(410, 'kết quả không còn (đã bị dọn sau JOB_TTL)')
    return FileResponse(path, media_type='video/mp4',
                        filename=os.path.basename(path))


print('API đã định nghĩa. Chạy cell dưới để bật server + tunnel.')

In [ ]:
# 1) Khởi động uvicorn trong thread nền (để cell còn chạy tiếp mở tunnel)
def _serve():
    uvicorn.run(api, host='0.0.0.0', port=PORT, log_level='warning')


threading.Thread(target=_serve, daemon=True).start()
time.sleep(3)

# 2) Mở cloudflared tunnel, đọc URL công khai từ chính log của nó
proc = subprocess.Popen(
    [CLOUDFLARED, 'tunnel', '--url', f'http://localhost:{PORT}', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

public_url = None
for line in proc.stdout:
    print(line, end='')
    m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
    if m and not public_url:
        public_url = m.group(0)
        # Dòng MỐC để script/bot bên ngoài đọc được URL:
        print('\n\nPUBLIC_URL=' + public_url + '\n', flush=True)
        break

print('=' * 66)
print('SẴN SÀNG —', public_url)
print()
print('  GET  /health              kiểm tra service')
print('  POST /swap                gửi việc, trả job_id ngay')
print('  GET  /jobs/{id}           poll trạng thái')
print('  GET  /jobs/{id}/result    tải mp4 về')
print()
print('Chưa xử lý video nào (KHÔNG tốn GPU cho tới khi có request thật).')
print('Giữ cell này chạy — đóng là mất tunnel.')
print('=' * 66)

# 3) Giữ cell sống: giữ tunnel + giữ phiên Colab khỏi bị ngắt vì idle
while True:
    line = proc.stdout.readline()
    if not line:
        break
    if 'ERR' in line or 'error' in line.lower():
        print(line, end='')

---
## Ghi chú

- **Chạy lại phần nào khi nào**: Colab còn kết nối thì chỉ chạy lại PHẦN C. Colab ngắt
  (thường sau ~90 phút không tương tác, hoặc hết 12 tiếng) thì mất sạch VRAM lẫn `/content`,
  phải chạy lại từ PHẦN A.
- **Muốn setup nhanh hơn ở lần sau**: mount Google Drive rồi đổi `MODEL_DIR` sang một thư mục
  trên Drive. `download()` đã có sẵn kiểm tra "file đủ lớn thì bỏ qua", nên lần sau không tải
  lại ~880 MB nữa. Riêng phần `pip install` thì vẫn phải chạy lại.
- **Nhiều request cùng lúc**: đã xử lý bằng `RUN_LOCK` ở PHẦN D — job đến sau xếp hàng đợi
  chứ không bị từ chối. Cần thiết vì `APP` / `SWAPPER` / `RESTORER` dùng chung và các model
  ONNX/PyTorch này không an toàn khi gọi song song, mà FastAPI lại chạy handler `def` trong
  threadpool. Lưu ý: xếp hàng nghĩa là job thứ hai đợi job thứ nhất xong — client cứ poll
  `/jobs/{id}` bình thường, không cần làm gì thêm.
- **Dọn file**: `JOB_TTL = 3600` tự xoá file kết quả + entry sau 1 giờ, mỗi khi có job mới
  vào. File trong `WORK_DIR` (ảnh/video tải về) thì **không** được dọn — chạy service nhiều
  ngày thì thêm bước dọn `/content/work`.
- **Nếu dùng PHẦN C (upload tay) thay vì API**: `process_video` gọi trực tiếp không đi qua
  `RUN_LOCK`, nhưng lúc đó chỉ có một mình bạn chạy nên không sao.
- **`do_swap=False, use_gfpgan=True`** vẫn phải chạy detect mỗi frame (để biết làm nét ở đâu),
  nên không nhanh hơn nhiều so với bật cả hai — phần đắt là GFPGAN chứ không phải swap.
- **Video nhiều người**: `pick_face()` luôn chọn mặt lớn nhất. Cần thay hai người khác nhau
  thì dùng `video_face_swap_kiss.ipynb` (có `FaceTracker` khớp danh tính ArcFace).
- **Flicker**: swap từng frame độc lập nên có thể giật nhẹ. GFPGAN tự nó cũng gây flicker vì
  "sáng tác" chi tiết khác nhau mỗi frame — `use_gfpgan=False` đôi khi lại đỡ giật hơn.